In [1]:
import scipy
import numpy as np
import pandas as pd

from datasets import load_dataset

from sklearn.utils import shuffle
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import euclidean_distances

import os
from pprint import pprint
from time import perf_counter as pf
from tqdm import tqdm

import torch
from torch import nn
from torch.nn import functional as F

d:\College 2\sem 4\Comp Bio\project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ds = load_dataset('tattabio/ec_classification')
# ds = load_dataset('tattabio/euk_retrieval')
train = ds['train']
test = ds['test']

In [3]:
train_entries = train.to_pandas()['Entry']
test_entries  = test.to_pandas()['Entry']

In [4]:
train

Dataset({
    features: ['Entry', 'Label', 'Sequence'],
    num_rows: 512
})

In [5]:
len(train_entries.loc[train_entries == 'A0AFT'].index)

0

In [6]:
def find_file(dir, uniprot_id):
    filenames = os.listdir(dir)
    for fname in filenames:
        if uniprot_id in fname:
            return pd.read_csv(os.path.join(dir, fname))
    
    return None

find_file('data/ec_classification/residue_centroids', 'Q59478')

,chain,seq,res_name,n_atoms,mean_x,mean_y,mean_z
0,A,1,MET,8,61.292,-47.118,34.188
1,A,2,LEU,8,57.516,-44.128,33.338
2,A,3,LYS,9,52.904,-47.440,33.570
3,A,4,SER,6,51.654,-42.406,31.942
4,A,5,GLY,4,47.866,-41.808,31.477
...,...,...,...,...,...,...,...
302,A,303,HIS,10,-2.169,-5.456,-16.419
303,A,304,GLY,4,-4.763,-6.814,-20.690
304,A,305,ALA,5,-7.114,-8.235,-23.256
305,A,306,GLN,9,-5.673,-12.276,-22.763


In [7]:
train_residue_centroids = {}
test_residue_centroids = {}
for row in tqdm(train):
    maybe_csv = find_file('data//ec_classification//residue_centroids//', row['Entry'])
    
    if maybe_csv is not None:
        train_residue_centroids[row['Entry']] = maybe_csv

for row in tqdm(test):
    maybe_csv = find_file('data//ec_classification//residue_centroids//', row['Entry'])
    
    if maybe_csv is not None:
        test_residue_centroids[row['Entry']] = maybe_csv

100%|██████████| 128/128 [00:00<00:00, 704.53it/s]


In [8]:
train_labels = []
for entry in train_residue_centroids.keys():
    idxrange = train_entries.loc[train_entries == entry].index
    if len(idxrange) > 0:
        train_labels.append(train[idxrange[0]]['Label'])

test_labels = []
for entry in test_residue_centroids.keys():
    idxrange = test_entries.loc[test_entries == entry].index
    if len(idxrange) > 0:
        test_labels.append(test[idxrange[0]]['Label'])

In [9]:
# find the label encoding
serieses = []
for key in train_residue_centroids.keys():
    serieses.append(train_residue_centroids[key].res_name)

for key in test_residue_centroids.keys():
    s = train_residue_centroids.get(key)
    if s is not None:
        serieses.append(s)

labenc = LabelEncoder()
print(pd.concat(serieses).unique())
ids = labenc.fit_transform(pd.concat(serieses).unique())
print(ids)

<ArrowStringArray>
['MET', 'ALA', 'THR', 'SER', 'ARG', 'LEU', 'ASN', 'CYS', 'PHE', 'PRO', 'ASP',
 'GLU', 'TYR', 'VAL', 'LYS', 'ILE', 'HIS', 'GLN', 'GLY', 'TRP']
Length: 20, dtype: str
[12  0 16 15  1 10  2  4 13 14  3  6 18 19 11  9  8  5  7 17]


In [10]:
def df_to_map(df):
    distances = euclidean_distances(df[['mean_x', 'mean_y', 'mean_z']])
    encoded_res = labenc.transform(df.res_name)
    c1 = np.repeat(np.array(encoded_res)[:, np.newaxis], repeats=len(encoded_res), axis=1)
    c2 = c1.T
    # c1 = F.one_hot(torch.tensor(c1), num_classes=len(ids))
    # c2 = F.one_hot(torch.tensor(c2), num_classes=len(ids))

    # return torch.concat([torch.unsqueeze(torch.tensor(distances), -1), c1, c2], dim = -1).permute(2, 0, 1).to(torch.float32)
    return torch.stack([torch.tensor(distances), torch.tensor(c1), torch.tensor(c2)]).to(torch.float32)


In [11]:
df_to_map(train_residue_centroids['A0A151EH88']).shape

torch.Size([3, 329, 329])

In [12]:
train_structures = []
for key in train_residue_centroids.keys():
    train_structures.append(df_to_map(train_residue_centroids[key]))

test_structures = []
for key in test_residue_centroids.keys():
    test_structures.append(df_to_map(test_residue_centroids[key]))

In [13]:
y_train = np.array(list(map(lambda x: int(x[0])-1, train_labels)))
y_test = np.array(list(map(lambda x: int(x[0])-1, test_labels)))

In [14]:
train_structures, y_train = shuffle(train_structures, y_train)
test_structures, y_test = shuffle(test_structures, y_test)

In [15]:
np.unique(y_test)

array([0, 1, 2, 3, 4, 5, 6])

In [16]:
# DEVICE = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
DEVICE = torch.device('cpu')

In [17]:
model = nn.Sequential(
    nn.Conv2d(3, 64, 3, padding=1),
    nn.ReLU(),
    nn.GroupNorm(num_groups=32, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    nn.GroupNorm(num_groups=32, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    nn.GroupNorm(num_groups=32, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    nn.GroupNorm(num_groups=32, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    nn.GroupNorm(num_groups=32, num_channels=64),

    nn.Conv2d(64, 64, 3, padding=1),
    nn.ReLU(),
    nn.GroupNorm(num_groups=32, num_channels=64),


    nn.AdaptiveAvgPool2d(1),
    nn.Flatten(1),
    nn.Linear(64, 7)
)
model.to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [18]:
torch.stack((train_structures[0], train_structures[0])).shape

torch.Size([2, 3, 140, 140])

In [19]:
y_pred = model(train_structures[0].unsqueeze(0).to(DEVICE))
# model(torch.stack((train_structures[0], train_structures[0])).to(DEVICE))

In [20]:
y_pred

tensor([[-0.0164, -0.0750,  0.1132, -0.0803, -0.0206,  0.1410,  0.0116]],
       grad_fn=<AddmmBackward0>)

In [21]:
accumulation_steps = 16

for epoch in range(3):
    for i, (X, y) in enumerate(zip(train_structures, y_train)):
        X = X.to(DEVICE).unsqueeze(0)
        y = torch.tensor(y).to(DEVICE).unsqueeze(0)
        y_pred = model(X)
        loss = criterion(y_pred, y.to(torch.long))
        loss = loss / accumulation_steps
        
        # Backward pass - gradients accumulate in .grad attributes
        loss.backward()
        
        # Only update weights every N steps
        if (i + 1) % accumulation_steps == 0:
            optimizer.step()        # Update weights
            optimizer.zero_grad()


In [22]:
model.eval()
total_loss = 0.0
correct = 0
total = 0

with torch.no_grad():
    for X, y in zip(test_structures, y_test):
        X = X.unsqueeze(0).to(DEVICE)
        y = torch.tensor(y).to(torch.long).unsqueeze(0).to(DEVICE)
        # print(y.shape)

        y_pred = model(X)
        loss = criterion(y_pred, y)
        total_loss += loss.cpu().item()

        # print(torch.max(y_pred))
        predicted = torch.argmax(y_pred.cpu())  # Shape: (1,)
        correct += (predicted == y).sum().item()
        total += 1
    
    avg_loss = total_loss / total
    accuracy = 100.0 * correct / total
    
print('avg_loss:', avg_loss)
print('accuracy:', accuracy)

avg_loss: 1.4864512884010703
accuracy: 44.067796610169495


In [176]:
torch.argmax(y_pred)

tensor(2)